# Objetivo del modelado

En este notebook se entrena un modelo de Machine Learning para pronosticar la demanda semanal por combinación tienda-producto.

Se utiliza una estrategia de validación temporal para evitar fuga de información y aproximar el escenario real de predicción.

In [24]:
import sys
from pathlib import Path

# Raíz del proyecto
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [53]:
import importlib
import src.evaluation
importlib.reload(src.evaluation)

<module 'src.evaluation' from 'c:\\Users\\beltr\\Documents\\prueba_tostao\\src\\evaluation.py'>

In [96]:
import importlib
import src.training
importlib.reload(src.training)

<module 'src.training' from 'c:\\Users\\beltr\\Documents\\prueba_tostao\\src\\training.py'>

In [104]:
import importlib
import src.forecasting
importlib.reload(src.forecasting)

<module 'src.forecasting' from 'c:\\Users\\beltr\\Documents\\prueba_tostao\\src\\forecasting.py'>

In [105]:
from sklearn.model_selection import TimeSeriesSplit
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import joblib

from src.config import PROCESSED_DATA
from src.training import *
from src.evaluation import *
from src.forecasting import *


In [73]:
df = load_training_data(PROCESSED_DATA / "training_dataset.csv")

In [74]:
df["fecha"] = pd.to_datetime(df["fecha"])

In [75]:
X, y = prepare_target(df)

In [80]:
train, test = split_data(df)

X_train, y_train = prepare_features_target(
    train,
    drop_columns=["fecha", "nombre"]
)
X_test, y_test = prepare_features_target(test)

In [48]:
preprocessor = build_preprocessor()

In [81]:
models = build_models(preprocessor)

results = compare_models(
    models,
    X_train,
    X_test,
    y_train,
    y_test
)

results

,Model,MAE,RMSE,MAPE,R2
2,Extra Trees,3.558327,4.689310,3.724852e+13,0.688725
1,Random Forest,3.566958,4.723011,3.795891e+13,0.684235
3,Gradient Boosting,3.574335,4.784536,3.512624e+13,0.675955
0,Linear Regression,3.669866,4.859900,3.003790e+13,0.665666


In [89]:
pipeline_extratrees = models["Extra Trees"]

grid = optimize_model(
    pipeline_extratrees,
    X_train,
    y_train
)

In [91]:
best_model = grid.best_estimator_

In [98]:
best_model = train_best_model(
    best_model,
    X_train,
    y_train
)

pred = predict(
    best_model,
    X_test
)

metrics = evaluate_metrics(
    y_test,
    pred
)

metrics

{'MAE': np.float64(3.492370371541575),
 'RMSE': np.float64(4.652412972584692),
 'MAPE': np.float64(37120853818500.39),
 'R2': 0.6936045578151846}

In [100]:
save_model(
    best_model,
    "../models/forecast_model.pkl"
)

In [101]:
future_forecast = forecast_next_week(
    history_df=df,
    model_path="../models/forecast_model.pkl"
)

In [106]:
save_forecast(
    future_forecast,
    "../data/processed/next_week_forecast.csv"
)